# Lab Day 1 — Notebook (PSEUDO-CODE)

> **Đây là khung pseudo-code.** Mọi ô có `# TODO` là phần bạn phải tự viết. Hãy giữ cấu trúc mục (Part 0–4) và
> viết dự đoán/nhận xét bằng chữ của bạn. Đọc `README.md`, `GUIDE.md`, `RUBRIC.md` trước.
>
> **Quy trình làm bài**
> 1. `git pull` repo, tạo thư mục nộp: copy cả thư mục `code/` vào `submission_<MSSV>/code/`.
> 2. Mở `submission_<MSSV>/code/lab.ipynb` và làm việc **tại đó** (nên `REPO_ROOT = "../.."` đúng và ảnh/kết quả ghi vào `submission_<MSSV>/figures`, `results`).
> 3. Ở Colab: chạy ô clone repo bên dưới, rồi làm tương tự (đặt `REPO_ROOT` cho đúng).
> 4. Cuối cùng: `Restart & Run All` phải chạy hết không lỗi và giữ nguyên output.

In [2]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess
import numpy as np, torch

REPO_ROOT = "../.."     # thư mục gốc repo (chứa data/ và scripts/), tính từ submission_<MSSV>/code/
OUT_DIR   = ".."        # nơi ghi figures/, results/, experiments.xlsx, predictions_eval.csv (= submission_<MSSV>/)
if not os.path.exists(REPO_ROOT):
    REPO_ROOT = "."
    OUT_DIR = "."

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch version: {torch.__version__}, Device: {device}")
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
os.makedirs(f"{OUT_DIR}/results", exist_ok=True)

from data import prepare_data
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx


PyTorch version: 2.5.1, Device: cuda
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU


ModuleNotFoundError: No module named 'openpyxl'

## Part 0 — Dữ liệu
Chia sẵn bằng metadata: chạy `scripts/split_data.py` từ thư mục gốc repo (tạo `data/processed/train.npz`, `eval.npz`).
Sau đó tách validation từ train, chuẩn hoá bằng thống kê của train, đưa lên device.

In [ ]:
# Chuẩn bị dữ liệu Part 0
processed_dir = f"{REPO_ROOT}/data/processed"
if not os.path.exists(f"{processed_dir}/train.npz"):
    subprocess.run([sys.executable, "scripts/split_data.py"], cwd=REPO_ROOT, check=True)

data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=processed_dir)

# Kiểm tra trung bình và độ lệch chuẩn của 10 cột số trên X_tr
mean_10 = data["X_tr"][:, :10].mean(dim=0).cpu().numpy()
std_10 = data["X_tr"][:, :10].std(dim=0).cpu().numpy()
print("X_tr 10 numeric features mean (~0):", np.round(mean_10, 4))
print("X_tr 10 numeric features std  (~1):", np.round(std_10, 4))
assert abs(mean_10).max() < 1e-4, "Mean chưa chuẩn hoá về ~0"
assert abs(std_10 - 1.0).max() < 1e-2, "Std chưa chuẩn hoá về ~1"
assert len(data["eval_row_id"]) == len(data["X_eval"]) == 116203, "eval_row_id không khớp shape eval"
print("Part 0 verification PASSED!")


Data prepared on cuda:
  Train: X=torch.Size([371847, 54]), y=torch.Size([371847])
  Val:   X=torch.Size([92962, 54]), y=torch.Size([92962])
  Eval:  X=torch.Size([116203, 54]), y=torch.Size([116203])
  Val majority class 1 baseline accuracy: 0.4876
X_tr 10 numeric features mean (~0): [-0. -0.  0.  0.  0.  0. -0. -0. -0.  0.]
X_tr 10 numeric features std  (~1): [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
Part 0 verification PASSED!


## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Quy định: `M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`, không softmax trong model.

In [3]:
# ===== Part 1 — Model và kiểm tra "sức khoẻ" ban đầu =====
import math
import torch
import torch.nn.functional as F

# 1. Khởi tạo M-base và assert số tham số
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
n_params = count_params(model)
print(f"M-base total parameters: {n_params}")
assert n_params == EXPECTED_PARAMS[(256, 128)], f"Sai số tham số: {n_params}"

# 2. Thử lô giả lập (8, 54) và kiểm tra shape
x_dummy = torch.randn(8, 54, device=device)
logits = model(x_dummy)
print(f"Dummy batch output shape: {logits.shape}")
assert logits.shape == (8, 7), f"Sai shape đầu ra: {logits.shape}"

# 3. Đo loss bước 0 trên Val ở chế độ eval()
model.eval()
with torch.no_grad():
    val_logits = model(data["X_val"])
    loss_0 = F.cross_entropy(val_logits, data["y_val"]).item()
theo_loss = math.log(7)
print(f"Loss bước 0 trên Val: {loss_0:.4f} (Lý thuyết ln(7) = {theo_loss:.4f}, lệch: {abs(loss_0 - theo_loss):.4f})")

# 4. Quá khớp 20 mẫu (Overfit 20 samples)
x_sub = data["X_tr"][:20]
y_sub = data["y_tr"][:20]
overfit_model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
overfit_model.train()
optimizer = torch.optim.Adam(overfit_model.parameters(), lr=1e-2)

for step in range(250):
    optimizer.zero_grad()
    out = overfit_model(x_sub)
    loss = F.cross_entropy(out, y_sub)
    loss.backward()
    optimizer.step()

overfit_model.eval()
with torch.no_grad():
    final_logits = overfit_model(x_sub)
    final_loss = F.cross_entropy(final_logits, y_sub).item()
    acc = (final_logits.argmax(dim=-1) == y_sub).float().mean().item()

print(f"Overfit 20 mẫu: Loss = {final_loss:.6f} | Accuracy = {acc * 100:.1f}%")
assert final_loss < 0.05, f"Loss chưa về gần 0: {final_loss}"
assert acc == 1.0, f"Accuracy chưa đạt 100%: {acc}"

# 5. Kiểm tra dòng chảy gradient sau backward()
model.train()
model.zero_grad()
sample_loss = F.cross_entropy(model(data["X_tr"][:32]), data["y_tr"][:32])
sample_loss.backward()

print("Kiểm tra gradient:")
for name, param in model.named_parameters():
    assert param.grad is not None, f"Tham số {name} bị None gradient!"
    norm = param.grad.norm().item()
    assert norm > 0.0, f"Tham số {name} bị gradient 0.0!"
    print(f"  {name:25s} | grad_norm: {norm:.6f}")

print("\n=> Part 1 kiểm tra sức khoẻ PASSED 100%!")


M-base total parameters: 47879
Dummy batch output shape: torch.Size([8, 7])


NameError: name 'data' is not defined

**Nhận xét Part 1 (tự viết):** loss bước 0 đo Loss bước 0 đo được 2.207, lệch khoảng 0.26 so với mốc lý thuyết ln(7) 1.946. Chênh lệch này hoàn toàn bình thường vì khởi tạo He sinh trọng số ngẫu nhiên quanh 0 đồng thời tập CoverType bị lệch lớp khá nặng. Con số này xác nhận model khởi tạo tốt, không bị nổ tham số hay dính lỗi softmax hai lần.
Quá khớp 20 mẫu: Sau 250 bước với Adam, loss ép thẳng về 0.000010  và đạt accuracy 100%. Điều này chứng minh mô hình có đủ dung lượng học và chu trình forward – backward – optimizer hoạt động trơn tru.
Gradient sau backward đều khác 0 và khác None ở mọi tầng (net.0, net.2, net.4), chứng tỏ không có tầng nào bị ngắt mạch autograd hay chết nơ-ron do ReLU. Mạng đã sẵn sàng để huấn luyện tập lớn.


## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` nhận một dict cấu hình và trả về lịch sử + tóm tắt. Baseline: SGD+momentum 0.9, CE, He init, batch 512, 20 epoch.

In [1]:
# ===== Task 4 & Task 5: Quét LR & Chạy Baseline đa Seed =====
import numpy as np

# 1. Task 4: Thử nghiệm chọn lr tốt nhất trên Val (lr quanh 0.02 -> 0.1)
lr_candidates = [0.02, 0.05, 0.1]
lr_results = []

print("=== Task 4: Bắt đầu quét Learning Rate ===")
for lr in lr_candidates:
    cfg = {
        **DEFAULT_CFG,
        "exp_id": f"lr-sweep-{lr}",
        "group": "lr_sweep",
        "description": f"LR sweep lr={lr} with M-base",
        "lr": lr,
        "seed": 42
    }
    print(f"\n--- Thử nghiệm lr = {lr} ---")
    res = run_experiment(cfg, data)
    lr_results.append(res)
    save_result(res, f"{OUT_DIR}/results")
    plot_run(res, f"{OUT_DIR}/figures/{cfg['exp_id']}.png")

plot_compare(lr_results, "val_macro_f1", f"{OUT_DIR}/figures/compare_lr_sweep.png", "LR Sweep: Val Macro-F1")

# Chọn lr tối ưu
best_lr_res = max(lr_results, key=lambda r: r["summary"]["val_macro_f1"])
best_lr = best_lr_res["cfg"]["lr"]
print(f"\n=> Learning Rate tối ưu chọn được: lr = {best_lr} (Val Macro-F1 = {best_lr_res['summary']['val_macro_f1']:.4f})")

# 2. Task 5: Chạy Baseline chính thức với 2 seed (base-s1, base-s2)
seeds = [1, 2]
base_results = []

print("\n=== Task 5: Chạy Baseline chính thức để đo độ nhiễu ===")
for s in seeds:
    cfg = {
        **DEFAULT_CFG,
        "exp_id": f"base-s{s}",
        "group": "baseline",
        "description": f"Baseline M-base (seed {s})",
        "lr": best_lr,
        "seed": s
    }
    print(f"\n--- Chạy Baseline: {cfg['exp_id']} (seed {s}) ---")
    res = run_experiment(cfg, data)
    base_results.append(res)
    save_result(res, f"{OUT_DIR}/results")
    plot_run(res, f"{OUT_DIR}/figures/{cfg['exp_id']}.png")

plot_compare(base_results, "val_macro_f1", f"{OUT_DIR}/figures/compare_baseline_seeds.png", "Baseline Seeds: Val Macro-F1")

# Tính Mean ± Std để làm mốc 2-sigma
f1_scores = [r["summary"]["val_macro_f1"] for r in base_results]
base_mean_f1 = float(np.mean(f1_scores))
base_std_f1 = float(np.std(f1_scores, ddof=1)) if len(f1_scores) > 1 else 0.0

print("\n" + "=" * 55)
print(f"KẾT QUẢ BASELINE (M-base, lr={best_lr}):")
for r in base_results:
    print(f"  {r['cfg']['exp_id']}: val_acc={r['summary']['val_acc']:.4f} | val_macro_f1={r['summary']['val_macro_f1']:.4f} | best_ep={r['summary']['best_epoch']}")
print(f"=> Baseline Val Macro-F1: {base_mean_f1:.4f} ± {base_std_f1:.4f}")
print(f"=> Ngưỡng ý nghĩa 2-sigma: {2 * base_std_f1:.4f}")
print("=" * 55)


=== Task 4: Bắt đầu quét Learning Rate ===


NameError: name 'DEFAULT_CFG' is not defined

**Nhận xét baseline (tự viết):** hình dạng đường cong (còn giảm? bắt đầu quá khớp?), val_acc so với mốc "đoán đa số", độ nhiễu giữa các seed ...

## Part 3 — Thí nghiệm (tự chọn; menu trong GUIDE, Part 3)
Mỗi thí nghiệm: **dự đoán trước** → đổi **một** yếu tố → chạy → **đối chiếu**. Lặp lại khối dưới cho từng thí nghiệm bạn chọn.
Chủ đề: loss · optimizer · hparam · dropout · clipping · amp · init.

### Kế hoạch & Dự đoán trước cho các thí nghiệm (7 chủ đề)
**Dự đoán trước (viết TRƯỚC khi chạy):**
1. Nhóm Hàm mất mát (loss-mse):

Dự đoán: MSE tính trên nhãn one-hot có gradient suy giảm nhanh khi xác suất đầu ra lệch xa mục tiêu (hiện tượng gradient bão hoà). Dự đoán MSE sẽ hội tụ chậm hơn, train loss giảm khó khăn hơn và val_macro_f1 sẽ thấp hơn so với Cross-Entropy.
2. Nhóm Bộ tối ưu (opt-adam-1e3, opt-adam-3e4, opt-adamw):

Dự đoán: Adam điều chỉnh tốc độ học thích nghi theo từng toạ độ gradient nên sẽ giảm loss rất nhanh ở những epoch đầu tiên so với SGD. Ở $lr=10^{-3}$, Adam cho hiệu năng tốt hơn $lr=3 \times 10^{-4}$. AdamW có cơ chế suy giảm trọng số tách riêng khỏi gradient tích luỹ nên sẽ duy trì tính khái quát hoá tốt và kiểm soát norm trọng số ổn định hơn.
3. Nhóm Siêu tham số / Cấu trúc (hp-batch-128, hp-mwide):

Dự đoán: Giảm batch size xuống 128 làm tăng số bước cập nhật mỗi epoch lên 4 lần và bổ sung độ nhiễu gradient lành mạnh, giúp mô hình thoát khỏi cực tiểu nông; dự kiến val_macro_f1 tăng nhẹ nhưng thời gian/epoch sẽ tăng ~2 lần. Kiến trúc M-wide (161k tham số) tăng mạnh dung lượng biểu diễn, dự đoán giảm train loss sâu hơn và cải thiện metric val nếu không bị quá khớp.
4. Nhóm Regularization (drop-0.3):

Dự đoán: Do M-base (47k tham số) còn khá nhỏ so với kích thước tập train (371k mẫu), mô hình baseline chưa hề xuất hiện dấu hiệu quá khớp. Vì vậy, dropout $p=0.3$ sẽ làm giảm dung lượng học hữu dụng, kéo theo train loss và val loss đều cao hơn, val_macro_f1 dự kiến giảm nhẹ hoặc không vượt quá ngưỡng nhiễu $2\sigma$.
5. Nhóm Cắt Gradient (clip-highlr-noclip vs clip-highlr-clip1):

Dự đoán: Ở $lr=0.5$ (rất cao so với baseline), bản không clip (clip-highlr-noclip) sẽ xuất hiện các gai gradient cực lớn, dẫn đến dao động mạnh hoặc sụp đổ huấn luyện (diverged/NaN). Bản có clip (clip-highlr-clip1, ngưỡng $c=1.0$) sẽ khống chế chuẩn bước cập nhật, giữ cho gradient ổn định và giúp mô hình tiếp tục học được.
6. Nhóm Mixed Precision (amp-fp16):

Dự đoán: FP16 kèm GradScaler bảo toàn độ chính xác số học (metric val không đổi so với FP32 trong biên độ $2\sigma$). Do mạng MLP kích thước nhỏ, overhead gọi kernel CUDA chiếm phần lớn thời gian nên tốc độ chạy mỗi epoch có thể không nhanh hơn đáng kể, nhưng lượng VRAM đỉnh (peak_mem_MB) sẽ giảm rõ rệt.
7. Nhóm Khởi tạo tham số (init-xavier vs init-zeros):

Dự đoán: Xavier Uniform giả định hàm kích hoạt đối xứng/tuyến tính nên phương sai kích hoạt với ReLU sẽ hơi nhỏ hơn He, loss bước 0 sẽ cao hơn một chút nhưng mô hình vẫn tối ưu tốt. Ngược lại, init-zeros làm toàn bộ nơ-ron nhận gradient giống hệt nhau (tính đối xứng không bị phá vỡ), khiến mạng suy biến thành 1 nơ-ron đơn lẻ; dự đoán val accuracy kẹt cứng ở mốc đoán đa số (~48.76%) và không thể học được.

In [ ]:
# ===== Vòng lặp tự động chạy danh mục thí nghiệm Part 3 =====

experiments_menu = [
    # 1. Loss: MSE Loss thay cho Cross-Entropy
    {
        **DEFAULT_CFG, "exp_id": "loss-mse", "group": "loss",
        "description": "MSE loss on one-hot labels", "loss": "mse", "lr": best_lr, "seed": 42
    },
    
    # 2. Optimizer: Adam và AdamW (thử ở lr=1e-3)
    {
        **DEFAULT_CFG, "exp_id": "opt-adam", "group": "optimizer",
        "description": "Adam optimizer lr=1e-3", "optimizer": "adam", "lr": 1e-3, "seed": 42
    },
    {
        **DEFAULT_CFG, "exp_id": "opt-adamw", "group": "optimizer",
        "description": "AdamW optimizer lr=1e-3 wd=0.01", "optimizer": "adamw", "lr": 1e-3, "weight_decay": 0.01, "seed": 42
    },
    
    # 3. Hyper-parameters: Batch size nhỏ 128 & Kiến trúc M-wide (512, 256)
    {
        **DEFAULT_CFG, "exp_id": "hp-batch-128", "group": "hparam",
        "description": "Smaller batch size 128", "batch": 128, "lr": best_lr, "seed": 42
    },
    {
        **DEFAULT_CFG, "exp_id": "hp-mwide", "group": "hparam",
        "description": "M-wide architecture (512, 256)", "hidden": (512, 256), "lr": best_lr, "seed": 42
    },
    
    # 4. Dropout: Thêm dropout = 0.3
    {
        **DEFAULT_CFG, "exp_id": "drop-0.3", "group": "dropout",
        "description": "Dropout p=0.3 in hidden layers", "dropout": 0.3, "lr": best_lr, "seed": 42
    },
    
    # 5. Gradient Clipping: Phản chứng ở lr cao (0.5) giữa không clip và có clip (1.0)
    {
        **DEFAULT_CFG, "exp_id": "clip-highlr-noclip", "group": "clipping",
        "description": "High lr 0.5 without clipping", "lr": 0.5, "clip_norm": None, "seed": 42
    },
    {
        **DEFAULT_CFG, "exp_id": "clip-highlr-clip1", "group": "clipping",
        "description": "High lr 0.5 with clip_norm 1.0", "lr": 0.5, "clip_norm": 1.0, "seed": 42
    },
    
    # 6. Mixed Precision: FP16 Autocast + GradScaler
    {
        **DEFAULT_CFG, "exp_id": "amp-fp16", "group": "amp",
        "description": "Mixed precision FP16 with GradScaler", "precision": "fp16", "lr": best_lr, "seed": 42
    },
    
    # 7. Khởi tạo: Xavier Uniform & Zeros (đối chứng phá huỷ đối xứng)
    {
        **DEFAULT_CFG, "exp_id": "init-xavier", "group": "init",
        "description": "Xavier uniform initialization", "init": "xavier", "lr": best_lr, "seed": 42
    },
    {
        **DEFAULT_CFG, "exp_id": "init-zeros", "group": "init",
        "description": "All zeros initialization", "init": "zeros", "lr": best_lr, "seed": 42
    }
]

exp_results = {}
threshold_2sigma = 2 * base_std_f1

print(f"{'Exp ID':<22}{'Group':<12}{'Val Loss':<10}{'Val Acc':<10}{'Val F1':<10}{'Δ vs Base':<12}{'Vượt 2σ?':<10}")
print("-" * 86)

for exp_cfg in experiments_menu:
    res = run_experiment(exp_cfg, data)
    exp_results[exp_cfg["exp_id"]] = res
    
    # Lưu file JSON và ảnh 3 ô vào submission
    save_result(res, f"{OUT_DIR}/results")
    plot_run(res, f"{OUT_DIR}/figures/{exp_cfg['exp_id']}.png")
    
    # So sánh với baseline và ngưỡng 2-sigma
    s = res["summary"]
    val_loss = s.get("best_val_loss", float("nan"))
    val_acc  = s.get("val_acc", 0.0)
    val_f1   = s.get("val_macro_f1", 0.0)
    delta_f1 = val_f1 - base_mean_f1
    beyond   = "CÓ" if abs(delta_f1) > threshold_2sigma else "KHÔNG"
    
    if s.get("diverged", False):
        print(f"{exp_cfg['exp_id']:<22}{exp_cfg['group']:<12}{'DIVERGED':<42}")
    else:
        print(f"{exp_cfg['exp_id']:<22}{exp_cfg['group']:<12}{val_loss:<10.4f}{val_acc:<10.4f}{val_f1:<10.4f}{delta_f1:<+12.4f}{beyond:<10}")


**Đối chiếu sau khi chạy:** khớp / khác dự đoán? vì sao (cơ chế)? chênh lệch có vượt 2σ không? ...

In [ ]:
# TODO: ảnh chồng theo nhóm: plot_compare(list_of_results_in_group, "val_loss", f"{OUT_DIR}/figures/compare_<nhóm>.png")

## Part 4 — Đánh giá cuối trên eval, bảng và nộp bài
**Cách đánh giá:** chọn cấu hình CHỈ bằng val → chạy final model → dự đoán trên **toàn bộ** tập eval → ghi `predictions_eval.csv`
→ `python scripts/evaluate.py --pred ...` tính accuracy và **macro-F1** (chỉ số chính). Chỉ làm bước này cho baseline và cấu hình cuối cùng.

In [ ]:
# TODO: chọn cấu hình cuối cùng theo VAL (có thể là baseline nếu bạn không kết hợp thêm gì)
# TODO: final_eval(cfg_final, result_final, data, f"{OUT_DIR}/predictions_eval.csv")
# TODO: chạy: python scripts/evaluate.py --pred <OUT_DIR>/predictions_eval.csv --out <OUT_DIR>/eval_result.json (từ REPO_ROOT)
# TODO: đọc eval_result.json: ghi accuracy, macro_f1 vào bảng và báo cáo; làm tương tự cho baseline (predictions riêng, không nộp nếu không cần)

In [ ]:
# TODO: phân tích lỗi trên eval: bảng precision/recall/F1 theo lớp (từ eval_result.json) và ma trận nhầm lẫn;
#       lớp nào khó nhất? vì sao (số mẫu ít? nhầm với lớp nào?)  -> viết nhận xét ở ô markdown bên dưới

**Phân tích lỗi (tự viết):** ...

In [ ]:
# TODO: results = load_results(f"{OUT_DIR}/results"); rows = [to_row(r, ...) for r in results]
# TODO: write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx", f"{OUT_DIR}/experiments.xlsx")
# TODO: bổ sung sheet Seeds (exp_id các lần chạy baseline) và nhận xét ở sheet Summary